# AI Security Guardrails — Research Notebook\n\n**Purpose.** Reproducibly inspect a layered, deterministic guardrail prototype for a healthcare-information assistant. This notebook provides a transparent experimental workflow—not a claim that the system is clinically safe or that a PhD-level study has been completed.\n\nIt records environment and dataset provenance, runs automated checks, evaluates the labelled JSONL examples, compares a permissive baseline with the configured guardrails, examines failure cases, and exports a machine-readable report. Dataset examples are synthetic and the labels require independent review.\n\n**Safety boundary:** do not enter real health information. These rules are heuristic, can miss dangerous content, and can block benign text. Do not use this project for diagnosis, triage, prescribing, or patient care.

## 1. Reproducible setup\nRun in a fresh Colab runtime. This notebook checks out the repository's `main` branch. For a publication-quality experiment, pin a reviewed commit SHA and record it in the final report rather than relying on a moving branch.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, platform, subprocess, datetime

REPO_URL = 'https://github.com/Dinakarnayak/ai-security-guardrails-starter.git'
REPO_DIR = Path('/content/ai-security-guardrails-starter')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repository directory already exists; recording its current checked-out revision.')
os.chdir(REPO_DIR)
commit_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Repository commit:', commit_sha)
print('Notebook timestamp UTC:', datetime.datetime.now(datetime.timezone.utc).isoformat())
assert (REPO_DIR / 'app.py').is_file(), 'app.py was not found; verify repository checkout.'
assert (REPO_DIR / 'evaluate_guardrails.py').is_file(), 'Evaluation script was not found.'

## 2. Study design and interpretation\n\nThe bundled benchmark is a **small synthetic smoke-test set**, not an independently adjudicated benchmark. Results describe agreement with these specific labels only. Before making research claims, create a larger ethically sourced set, write a labelling rubric, use independent annotators, resolve disagreements, freeze train/development/test splits, and evaluate only once on the held-out test set.\n\nThis notebook compares the configured guardrail pipeline to a permissive baseline that allows everything. The baseline is intentionally simple; it is not a competing model. The comparison does not establish generalisation, causal benefit, or clinical utility.

## 3. Run regression checks\nThis verifies known policy behaviours before evaluation. A passing self-test is not a proof of complete safety.

In [ ]:
self_test = subprocess.run([sys.executable, 'app.py', '--self-test'], text=True, capture_output=True)
print(self_test.stdout)
if self_test.stderr:
    print(self_test.stderr)
assert self_test.returncode == 0, 'Self-test failed; stop and investigate before interpreting the benchmark.'

## 4. Validate and fingerprint the dataset\nThe SHA-256 fingerprint supports reproducibility. It does not establish the correctness of any label.

In [ ]:
cases_path = Path('evaluation/cases.jsonl')
raw_cases = cases_path.read_bytes()
case_lines = [line for line in raw_cases.decode('utf-8').splitlines() if line.strip()]
cases = [json.loads(line) for line in case_lines]
required = {'id', 'stage', 'text', 'expected_action', 'expected_category', 'risk_label', 'rationale'}
assert cases, 'Benchmark contains no cases.'
assert all(required <= set(c) for c in cases), 'At least one case lacks required fields.'
assert len({c['id'] for c in cases}) == len(cases), 'Case IDs must be unique.'
assert all(c['stage'] in {'input', 'output'} for c in cases), 'Unsupported stage found.'
assert all(c['risk_label'] in {0, 1} for c in cases), 'risk_label must be 0 or 1.'
dataset_sha256 = hashlib.sha256(raw_cases).hexdigest()
from collections import Counter
print('Cases:', len(cases))
print('Stage counts:', dict(Counter(c['stage'] for c in cases)))
print('Label counts:', dict(Counter(c['risk_label'] for c in cases)))
print('SHA-256:', dataset_sha256)
print('Reminder: inspect each text, expected label, and rationale before reporting metrics.')

## 5. Evaluate configured guardrails\nRun the repository evaluation harness and save the case-level JSON report. Metric values are descriptive for this benchmark only.

In [ ]:
results_path = Path('evaluation/colab_results.json')
evaluation = subprocess.run([sys.executable, 'evaluate_guardrails.py', '--cases', str(cases_path), '--output', str(results_path)], text=True, capture_output=True)
print(evaluation.stdout)
if evaluation.stderr:
    print(evaluation.stderr)
assert evaluation.returncode == 0, 'Evaluation harness returned a non-zero exit code.'
guardrail_report = json.loads(results_path.read_text(encoding='utf-8'))
print('Saved report:', results_path)

## 6. Compare to a permissive baseline\nFor this descriptive comparison, the baseline allows every example. Positive cases are therefore missed; benign cases are not falsely blocked. The guarded system uses `action != ALLOW` as its risk decision, so escalations, blocks and replacements count as risk interventions.

In [ ]:
from app import inspect_input, validate_output
def classify_guardrail(case):
    decision = inspect_input(case['text']) if case['stage'] == 'input' else validate_output(case['text'])
    return int(decision.action != 'ALLOW'), decision.action, decision.category

def binary_metrics(labels, predictions):
    tp = sum(y == 1 and p == 1 for y, p in zip(labels, predictions))
    tn = sum(y == 0 and p == 0 for y, p in zip(labels, predictions))
    fp = sum(y == 0 and p == 1 for y, p in zip(labels, predictions))
    fn = sum(y == 1 and p == 0 for y, p in zip(labels, predictions))
    div = lambda a, b: (a / b if b else None)
    return {'n': len(labels), 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
            'accuracy': div(tp + tn, len(labels)), 'precision': div(tp, tp + fp),
            'recall': div(tp, tp + fn), 'false_positive_rate': div(fp, fp + tn),
            'false_negative_rate': div(fn, fn + tp)}

labels = [int(c['risk_label']) for c in cases]
predictions, actions, categories = [], [], []
for c in cases:
    pred, action, category = classify_guardrail(c)
    predictions.append(pred); actions.append(action); categories.append(category)
baseline_predictions = [0] * len(labels)
comparison = {'permissive_baseline': binary_metrics(labels, baseline_predictions),
              'configured_guardrails': binary_metrics(labels, predictions),
              'interpretation': 'Synthetic benchmark only; not generalisation or clinical safety evidence.'}
print(json.dumps(comparison, indent=2))

## 7. Uncertainty and error analysis\nThe next cell reports a Wilson 95% interval for the observed binary accuracy. With a tiny, non-random synthetic set, this interval is only a rough description of sampling uncertainty; it does not fix dataset bias or label uncertainty.

In [ ]:
import math
def wilson_interval(successes, n, z=1.959963984540054):
    if n == 0: return (None, None)
    p = successes / n; z2 = z*z; denom = 1 + z2/n
    centre = (p + z2/(2*n))/denom
    margin = z * math.sqrt((p*(1-p)/n) + z2/(4*n*n)) / denom
    return (max(0.0, centre-margin), min(1.0, centre+margin))
correct = sum(y == p for y, p in zip(labels, predictions))
print('Configured risk-label accuracy:', correct, '/', len(labels))
print('Wilson 95% interval:', wilson_interval(correct, len(labels)))

errors = []
for c, pred, action, category in zip(cases, predictions, actions, categories):
    expected_action = c['expected_action']
    actual_action = action
    expected_category = c['expected_category']
    if actual_action != expected_action or category != expected_category or pred != c['risk_label']:
        errors.append({'id': c['id'], 'stage': c['stage'], 'text': c['text'],
                       'expected_action': expected_action, 'actual_action': actual_action,
                       'expected_category': expected_category, 'actual_category': category,
                       'risk_label': c['risk_label'], 'predicted_risk': pred,
                       'rationale': c['rationale']})
print('Cases with action/category/risk-label disagreement:', len(errors))
for error in errors:
    print(json.dumps(error, ensure_ascii=False))

## 8. Threats to validity and next research steps\n\nBefore this can support a defensible research paper or dissertation, the following work is still needed:\n\n1. **Construct validity:** define what counts as unsafe, appropriate escalation, overblocking, and unsafe output, with operational labelling rules.\n2. **Independent annotation:** use at least two qualified annotators where appropriate, measure agreement, document adjudication and retain label provenance.\n3. **Dataset validity:** expand coverage across benign, risky, ambiguous, paraphrased, adversarial, misspelled and (if in scope) multilingual prompts; preserve a held-out set and prevent leakage. Do not use real patient data without ethics and governance approvals.\n4. **Experimental controls:** compare input-only, output-only and layered configurations; measure latency and memory; repeat relevant stochastic model experiments with fixed seeds where supported.\n5. **Statistical reporting:** report denominators and confidence intervals by category, examine cost-sensitive errors, and avoid claims based on a tiny test set.\n6. **External validity:** evaluate on independently sourced data and, if a model is introduced, multiple model versions. A deterministic regex smoke test is not a substitute for evaluating model interactions.\n\nPatterns in this prototype can be evaded through paraphrase, negation, context, typographical variation or new forms of attack. This notebook is an educational evaluation scaffold, not a clinical safety mechanism.

## 9. Export a reproducibility manifest\nThe manifest records environment, repository revision, benchmark fingerprint, run time and summaries. It intentionally excludes raw prompts from the manifest itself, though the optional case-level report contains synthetic benchmark text.

In [ ]:
manifest = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'repository_url': REPO_URL,
    'repository_commit': commit_sha,
    'python_version': sys.version,
    'platform': platform.platform(),
    'dataset_path': str(cases_path),
    'dataset_sha256': dataset_sha256,
    'case_count': len(cases),
    'comparison': comparison,
    'accuracy_wilson_95': wilson_interval(correct, len(labels)),
    'guardrail_harness_overall': guardrail_report.get('overall'),
    'limitations': ['Synthetic cases only', 'Labels not independently validated', 'Pattern-based rules', 'Not for clinical use']
}
manifest_path = Path('evaluation/colab_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2) + chr(10), encoding='utf-8')
print(json.dumps(manifest, indent=2))
print('Manifest saved to', manifest_path)

## Coursework reminder\n\nFollow your module's latest assessment brief. If the assessment requires local laptop execution and a continuous Teams recording, this optional Colab notebook does not replace that requirement. Only use Colab if the convenor has approved it. Be ready to explain and defend every part of your implementation.